# SD 1.5 LoRA for pixel-art sprites (Colab T4)
Run the cells top to bottom. Your dataset must be on Google Drive at `MyDrive/lora/tinyrpg_dataset/img/10_tnyrpg/`.
Use a **T4 GPU** runtime: Runtime > Change runtime type.
Rule: we do not downgrade or pin anything that Colab already has.

## 1. Check the GPU and connect Google Drive
We need a GPU. Drive keeps your dataset and checkpoints safe if Colab disconnects.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

## 2. Install diffusers and get the training script
We install the latest `diffusers`. Then we download the training script from the **same version tag**, so the script and the library match.
We do not install the script's `requirements.txt` (it could downgrade Colab packages). The script only imports `datasets`, `peft`, `transformers`, `accelerate`, `torchvision`, `huggingface_hub`, `packaging`. Colab already has them; the last line checks this.

In [ ]:
!pip install -q -U diffusers
# Colab's torchao is older than new diffusers expects (ImportError FqnToConfig). We don't quantize, so remove it.
!pip uninstall -y -q torchao
import subprocess
ver = subprocess.check_output(["python", "-c", "import diffusers;print(diffusers.__version__)"], text=True).strip()
print("diffusers", ver)
!wget -q -O /content/train_text_to_image_lora.py https://raw.githubusercontent.com/huggingface/diffusers/v{ver}/examples/text_to_image/train_text_to_image_lora.py
!python -c "import datasets, peft, transformers, accelerate, torchvision, huggingface_hub, packaging; print('script deps ok')"
!ls -l /content/train_text_to_image_lora.py

## 3. Settings
Change values here only. Images are 512x512 pixel art, so no flip (sprites must keep their direction).

In [ ]:
DATA_SRC   = "/content/drive/MyDrive/lora/tinyrpg_dataset/img/10_tnyrpg"
LOCAL_DS   = "/content/ds"
OUTPUT_DIR = "/content/drive/MyDrive/lora/output_diffusers"
BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
RESOLUTION = 512
RANK = 32
LR = 1e-4
BATCH = 2
MAX_STEPS = 1000        # small test run. Raise for a real run.
CHECKPOINT_EVERY = 200  # a checkpoint on Drive every 200 steps
SEED = 42

## 4. Build the dataset
Copies the pictures to fast local disk and writes `metadata.jsonl` (picture name + caption) in the format the script reads.

In [ ]:
import os, json, shutil, glob
os.makedirs(LOCAL_DS, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
n = 0
with open(f"{LOCAL_DS}/metadata.jsonl", "w") as f:
    for png in sorted(glob.glob(f"{DATA_SRC}/*.png")):
        stem = os.path.splitext(os.path.basename(png))[0]
        shutil.copy(png, f"{LOCAL_DS}/{stem}.png")
        text = open(f"{DATA_SRC}/{stem}.txt").read().strip()
        f.write(json.dumps({"file_name": f"{stem}.png", "text": text}) + "\n")
        n += 1
print(n, "pairs")

## 5. Train
Runs the official diffusers LoRA script. `--resume_from_checkpoint latest` means: after a disconnect, run Settings and this cell again and it continues from the newest checkpoint on Drive (`MAX_STEPS` is the total, not the rest).
We use plain AdamW (this script version has no 8-bit option). It fits on a T4 for SD 1.5 rank 32 with fp16 and gradient checkpointing.

In [ ]:
!accelerate launch --mixed_precision fp16 /content/train_text_to_image_lora.py \
  --pretrained_model_name_or_path={BASE_MODEL} \
  --train_data_dir={LOCAL_DS} --image_column image --caption_column text \
  --resolution {RESOLUTION} --image_interpolation_mode nearest \
  --rank {RANK} --learning_rate {LR} --lr_scheduler cosine --lr_warmup_steps 50 \
  --train_batch_size {BATCH} --max_train_steps {MAX_STEPS} \
  --checkpointing_steps {CHECKPOINT_EVERY} --resume_from_checkpoint latest \
  --output_dir {OUTPUT_DIR} --mixed_precision fp16 --gradient_checkpointing \
  --seed {SEED} --dataloader_num_workers 2

## 6. Try it
Loads the base model plus your LoRA, makes 8 pictures with fixed seeds, saves them to `OUTPUT_DIR/samples` and shows a grid.
If this cell errors about memory, Runtime > Restart session, run Settings, then this cell.

In [ ]:
import torch, os
from diffusers import StableDiffusionPipeline
from PIL import Image
pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, safety_checker=None).to("cuda")
pipe.load_lora_weights(OUTPUT_DIR)
tail = "side view, full body, plain background"
items = [("knight with blue cape","idle"),("knight with blue cape","attack"),("soldier","walk"),("orc","idle"),
         ("archer","idle"),("mage","attack"),("goblin","walk"),("demon a","idle")]
os.makedirs(f"{OUTPUT_DIR}/samples", exist_ok=True)
imgs = []
for i, (x, a) in enumerate(items):
    g = torch.Generator("cuda").manual_seed(100 + i)
    im = pipe(f"tnyrpg, pixel art, {x}, {a}, {tail}", num_inference_steps=30, guidance_scale=7, generator=g).images[0]
    im.save(f"{OUTPUT_DIR}/samples/sample_{i}.png"); imgs.append(im)
grid = Image.new("RGB", (512 * 4, 512 * 2))
for i, im in enumerate(imgs): grid.paste(im, ((i % 4) * 512, (i // 4) * 512))
grid.save(f"{OUTPUT_DIR}/samples/grid.png")
display(grid.resize((1024, 512)))